# Ivy — IT Helpdesk Agent · Evaluation Scaffold (LangSmith-integrated)

Route an IT request → answer from the KB (RAG) → or escalate. Then grade it on **5 metrics**:
routing F1, faithfulness, answer correctness, escalation correctness, latency/cost.

**LangSmith is wired in three ways** (the Week 4 requirements):
1. **Tracing** — every LLM call + the full pipeline is traced (Section 2 + `@traceable`).
2. **Dataset** — the 30 golden cases are uploaded as a LangSmith Dataset (Section 8).
3. **Evaluation** — `evaluate()` records scored experiments you compare baseline vs v2 in the UI (Sections 9 & 12).

> ⚠️ Scaffold. **TODO** cells are where you iterate. One model call per row, so a full run is a few minutes.

## 1. Install

In [ ]:
%pip install --quiet langchain langchain-openai langchain-community faiss-cpu \
    pydantic pandas scikit-learn python-dotenv langsmith tqdm
# Optional standard RAG metrics (version-brittle; the custom judge in Section 6/10 is the default):
# %pip install --quiet ragas

## 2. Config & keys — *LangSmith tracing turns on here*
Fill in `.env` (same folder) before running:
```
OPENAI_API_KEY=sk-...
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=ivy-it-evals
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
```

In [ ]:
import os, time, json
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"),   "Set OPENAI_API_KEY in .env first."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY in .env (LangSmith is required for this project)."

# Tracing on — every LangChain LLM call below is now traced into your project.
os.environ["LANGSMITH_TRACING"]  = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "ivy-it-evals")
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT

# --- knobs ---
ROUTER_MODEL = "gpt-4o-mini"
JUDGE_MODEL  = "gpt-4o"
EMBED_MODEL  = "text-embedding-3-small"
TEMPERATURE  = 0
TOP_K        = 4
CONF_THRESHOLD = 0.55
CHUNK_SIZE, CHUNK_OVERLAP = 500, 75
KB_DIR       = "ivy_kb"
DATASET_CSV  = "ivy_golden_dataset.csv"
print("Config loaded. LangSmith project:", LANGSMITH_PROJECT)

## 2b. Verify the LangSmith connection

In [ ]:
from langsmith import Client
ls_client = Client()
# Touches the API so a bad key fails loudly now, not 20 minutes in.
_ = list(ls_client.list_datasets(limit=1))
print("LangSmith connected. Open your traces at https://smith.langchain.com  (project:", LANGSMITH_PROJECT, ")")

## 3. Define intents — *the descriptions ARE the prompt*
Confusable pair to watch: **software_access** ("I need access") vs **escalate** ("my access broke / I'm compromised").

In [ ]:
from enum import Enum

class Intent(str, Enum):
    password_reset   = "password_reset"
    vpn_connectivity = "vpn_connectivity"
    software_access  = "software_access"
    hardware_howto   = "hardware_howto"
    escalate         = "escalate"
    out_of_scope     = "out_of_scope"

INTENT_DESCRIPTIONS_V1 = {
    "password_reset":   "User wants to reset, change, or recover a forgotten/expired password, or is locked out from failed sign-in attempts.",
    "vpn_connectivity": "User needs help connecting to or setting up the company VPN / remote access.",
    "software_access":  "User is REQUESTING new access to an app, license, or shared drive, or an installed app won't open.",
    "hardware_howto":   "User needs how-to help with hardware they can self-fix: monitors, printers, projectors, peripherals.",
    "escalate":         "Anything needing a human: hardware failure, a security incident or suspected compromise, lost MFA device, or access that PREVIOUSLY worked and suddenly broke. Also any request to bypass policy or reveal credentials.",
    "out_of_scope":     "Not an IT request, or a personal/home device or personal account (home router, personal Gmail), or non-IT topics (weather, travel).",
}
def intent_block(desc): return "\n".join(f"- {k}: {v}" for k, v in desc.items())
print(intent_block(INTENT_DESCRIPTIONS_V1))

## 4. Build the knowledge base → chunk → embed → FAISS

In [ ]:
import glob
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

paths = sorted(glob.glob(os.path.join(KB_DIR, "*.md")))
assert paths, f"No .md files found in {KB_DIR}/"
raw_docs = [Document(page_content=open(p, encoding="utf-8").read(),
                     metadata={"source": os.path.basename(p)}) for p in paths]
chunks = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP).split_documents(raw_docs)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"{len(raw_docs)} docs -> {len(chunks)} chunks indexed.")

## 5. Router + judge models (structured output)

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class RouterOutput(BaseModel):
    intent: Intent = Field(description="The single best intent for this request.")
    confidence: float = Field(description="0-1 confidence in the chosen intent.")
    reasoning: str = Field(description="One sentence: the cue you routed on.")

class Faith(BaseModel):
    faithful: bool = Field(description="True if EVERY claim in the answer is supported by the context.")
    reasoning: str

class Correct(BaseModel):
    correct: bool = Field(description="True if the answer matches the ground-truth answer in substance.")
    reasoning: str

router_llm  = ChatOpenAI(model=ROUTER_MODEL, temperature=TEMPERATURE).with_structured_output(RouterOutput)
answer_llm  = ChatOpenAI(model=ROUTER_MODEL, temperature=TEMPERATURE)
faith_llm   = ChatOpenAI(model=JUDGE_MODEL,  temperature=0).with_structured_output(Faith)
correct_llm = ChatOpenAI(model=JUDGE_MODEL,  temperature=0).with_structured_output(Correct)
print("models ready")

## 6. The Ivy pipeline — `@traceable` so route→retrieve→answer is ONE trace

In [ ]:
from langsmith import traceable

ROUTER_PROMPT = """You are Ivy, an IT helpdesk router. Classify the user's request into exactly one intent.

Intents:
{intents}

Rules:
- Route on the user's PRIMARY request, not surface keywords.
- "I need access" = software_access. "Access that USED TO work and now fails" = escalate.
- Personal/home devices or accounts = out_of_scope.

User request: {query}"""

ANSWER_PROMPT = """You are Ivy, an IT helpdesk assistant. Answer ONLY from the context below.
If the context doesn't cover it, say you'll route the user to a human. Be concise.

Context:
{context}

User: {query}
Answer:"""

@traceable(run_type="chain", name="ivy")
def run_ivy(query, descriptions=INTENT_DESCRIPTIONS_V1):
    r = router_llm.invoke(ROUTER_PROMPT.format(intents=intent_block(descriptions), query=query))
    escalated = (r.intent == Intent.escalate) or (r.confidence < CONF_THRESHOLD)
    if r.intent == Intent.out_of_scope:
        return dict(intent=r.intent.value, confidence=r.confidence, reasoning=r.reasoning,
                    answer="That's outside IT support — I can't help with that here.", contexts=[], escalated=False)
    if escalated:
        return dict(intent=r.intent.value, confidence=r.confidence, reasoning=r.reasoning,
                    answer="This needs a human — I'm escalating you to the IT/security team.", contexts=[], escalated=True)
    docs = retriever.invoke(query)
    ctx = "\n\n".join(d.page_content for d in docs)
    ans = answer_llm.invoke(ANSWER_PROMPT.format(context=ctx, query=query)).content
    return dict(intent=r.intent.value, confidence=r.confidence, reasoning=r.reasoning,
                answer=ans, contexts=[d.page_content for d in docs], escalated=False)

## 7. Sanity check — test ONE before testing all (then check it appears in LangSmith)

In [ ]:
from pprint import pprint
pprint(run_ivy("How do I connect to the company VPN from home?"))
print("---")
pprint(run_ivy("I used to have access to the finance dashboard but now I get permission denied."))
print("\n^ Open LangSmith now — you should see two 'ivy' traces with nested router/retriever/answer steps.")

## 8. Upload the golden dataset to LangSmith
The Week 4 rule: *store the golden set as a LangSmith Dataset, not a loose CSV* — so you can re-run and compare versions.

In [ ]:
import pandas as pd
gold = pd.read_csv(DATASET_CSV)
gold["should_escalate"] = gold["should_escalate"].astype(str).str.upper().eq("TRUE")

LS_DATASET = "ivy-golden-v1"
existing = [d for d in ls_client.list_datasets() if d.name == LS_DATASET]
if existing:
    ls_dataset = existing[0]; print("dataset already exists:", LS_DATASET)
else:
    ls_dataset = ls_client.create_dataset(LS_DATASET, description="Ivy IT helpdesk — 30 labeled cases")
    ls_client.create_examples(
        inputs =[{"user_input": r.user_input} for r in gold.itertuples()],
        outputs=[{"target_intent": r.target_intent, "should_escalate": bool(r.should_escalate),
                  "ground_truth_answer": r.ground_truth_answer, "scenario_type": r.scenario_type}
                 for r in gold.itertuples()],
        dataset_id=ls_dataset.id,
    )
    print("uploaded", len(gold), "examples to", LS_DATASET)

## 9. Run the evaluation through LangSmith (baseline)
Records a scored **experiment** under your project with full traces. Four evaluators:
routing & escalation (code-based), faithfulness & correctness (LLM-judge).

In [ ]:
from langsmith import evaluate

def ivy_target_v1(inputs: dict) -> dict:
    return run_ivy(inputs["user_input"], INTENT_DESCRIPTIONS_V1)

def ev_routing(run, example):
    return {"key": "routing_correct",
            "score": int(run.outputs.get("intent") == example.outputs.get("target_intent"))}

def ev_escalation(run, example):
    return {"key": "escalation_correct",
            "score": int(bool(run.outputs.get("escalated")) == bool(example.outputs.get("should_escalate")))}

def ev_faithful(run, example):
    ctx = "\n\n".join(run.outputs.get("contexts") or [])
    if not ctx:                      # escalation/refusal -> not applicable; skip
        return None
    f = faith_llm.invoke(f"Context:\n{ctx}\n\nAnswer:\n{run.outputs.get('answer')}\n\nIs every claim supported by the context?")
    return {"key": "faithful", "score": int(f.faithful)}

def ev_correct(run, example):
    if not (run.outputs.get("contexts")):  # only grade answered rows
        return None
    c = correct_llm.invoke(f"Ground truth:\n{example.outputs.get('ground_truth_answer')}\n\nAnswer:\n{run.outputs.get('answer')}\n\nDoes the answer match in substance?")
    return {"key": "correct", "score": int(c.correct)}

baseline = evaluate(
    ivy_target_v1, data=LS_DATASET,
    evaluators=[ev_routing, ev_escalation, ev_faithful, ev_correct],
    experiment_prefix="ivy-baseline", client=ls_client,
)
print("Baseline experiment created — open it in LangSmith to see per-metric scores + traces.")
# Note: if your langsmith version rejects a None return, change the skips to {"key":..., "score":1, "comment":"n/a"}.

## 10. (Local quick-look) Baseline metrics for your writeup
LangSmith is the system of record; these give you the confusion matrix + failure table fast, locally.

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def run_all(descriptions, label):
    rows = []
    for _, row in tqdm(gold.iterrows(), total=len(gold), desc=label):
        t0 = time.time(); out = run_ivy(row["user_input"], descriptions)
        out["latency_s"] = time.time() - t0; out["test_id"] = row["test_id"]; rows.append(out)
    res = pd.DataFrame(rows).merge(gold, on="test_id")
    res["intent_correct"]   = res["intent"] == res["target_intent"]
    res["escalate_correct"] = res["escalated"] == res["should_escalate"]
    return res

results_v1 = run_all(INTENT_DESCRIPTIONS_V1, "baseline-local")
results_v1.to_csv("ivy_results_v1.csv", index=False)
print("latency p50=%.2fs p95=%.2fs" % (np.percentile(results_v1.latency_s,50), np.percentile(results_v1.latency_s,95)))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
labels = [i.value for i in Intent]
print(classification_report(results_v1["target_intent"], results_v1["intent"], labels=labels, zero_division=0))
print("confusion (rows=true, cols=pred):", labels)
print(confusion_matrix(results_v1["target_intent"], results_v1["intent"], labels=labels))
print("escalation accuracy:", round(results_v1["escalate_correct"].mean(),3))

## 11. Failure analysis — cluster, don't list
Tag each failing row: `Intent_Confusion | Retrieval_Miss | Ungrounded_Answer | Missed_Escalation`.

In [ ]:
fails = results_v1[~results_v1["intent_correct"] | ~results_v1["escalate_correct"]]
print(len(fails), "failing rows")
fails[["test_id","scenario_type","user_input","target_intent","intent","should_escalate","escalated","reasoning"]]

## 12. Improvement loop (v2) — one fix per lever, then re-run in LangSmith
1. **Prompt:** sharpen the descriptions below. 2. **Retrieval:** change TOP_K/CHUNK_SIZE, rebuild Section 4. 3. **Guardrail:** tune CONF_THRESHOLD (Section 2).
Re-running `evaluate()` with a new prefix lets you use the LangSmith **Comparison view**.

In [ ]:
INTENT_DESCRIPTIONS_V2 = dict(INTENT_DESCRIPTIONS_V1)
INTENT_DESCRIPTIONS_V2["software_access"] = ("User is requesting access they do NOT currently have "
    "(new app, license, or drive), or an installed app fails to launch. NOT for access that recently stopped working.")
INTENT_DESCRIPTIONS_V2["escalate"] = (INTENT_DESCRIPTIONS_V1["escalate"] +
    " Strong signal: phrases like 'used to work', 'worked yesterday', 'now denied', 'locked out + suspicious activity'.")

def ivy_target_v2(inputs: dict) -> dict:
    return run_ivy(inputs["user_input"], INTENT_DESCRIPTIONS_V2)

v2 = evaluate(ivy_target_v2, data=LS_DATASET,
              evaluators=[ev_routing, ev_escalation, ev_faithful, ev_correct],
              experiment_prefix="ivy-v2", client=ls_client)

# local delta for the writeup
from sklearn.metrics import f1_score
results_v2 = run_all(INTENT_DESCRIPTIONS_V2, "v2-local")
f1 = lambda df: f1_score(df["target_intent"], df["intent"], average="macro", labels=labels)
print(f"routing F1  v1={f1(results_v1):.3f} -> v2={f1(results_v2):.3f}")
print(f"escalation  v1={results_v1.escalate_correct.mean():.3f} -> v2={results_v2.escalate_correct.mean():.3f}")
print("Now open LangSmith → Experiments → select ivy-baseline + ivy-v2 → Compare.")

## 13. LLM-judge alignment — trust, then verify
Report **Precision/Recall/F1 on the rare class, NOT accuracy** (failures are ~10%, accuracy lies).

In [ ]:
# 1) Hand-label ~25 answered rows for faithfulness in results_v1 as column 'human_faithful' (True/False).
# 2) Run the same judge locally and compare:
# answered = results_v1[results_v1["contexts"].map(bool)]
# answered["judge_faithful"] = [ ... call faith_llm per row ... ]
# from sklearn.metrics import classification_report, confusion_matrix
# print(confusion_matrix(answered["human_faithful"], answered["judge_faithful"]))
# print(classification_report(answered["human_faithful"], answered["judge_faithful"], zero_division=0))
# 3) Swap JUDGE_MODEL, re-run, keep the judge whose misses are easiest to defend.
print("Fill in human_faithful labels, then run the comparison above.")

## 14. Wrap-up — what LangSmith now gives you
- **Traces:** every `ivy` run with nested router/retriever/answer, latency, and **token cost per run**.
- **Dataset:** `ivy-golden-v1`, versionable and re-runnable.
- **Experiments:** `ivy-baseline` vs `ivy-v2` side-by-side in the Comparison view.

### Report checklist
- [ ] Baseline scores (routing F1, faithfulness, correctness, escalation, p50/p95, $/run)
- [ ] Confusion matrix + named failure clusters
- [ ] 3 improvements: lever, change, predicted vs **measured** delta (incl. one honest regression)
- [ ] Judge alignment: confusion matrix + rare-class P/R/F1 + model note
- [ ] **LangSmith Comparison link** (ivy-baseline vs ivy-v2) + a short Loom